# LAB-D2-04: PyTorch Autograd - Break It and Fix It

**Purpose:** Rebuild the NumPy workflow with PyTorch, map automated operations to mechanics, and repair one training or evaluation defect from evidence.

**Objectives:** `OBJ-D2-01`, `OBJ-D2-02`, `OBJ-D2-05`, `OBJ-D2-07`, `OBJ-D2-08`  
**Estimated duration:** 55 minutes live; total reference execution under 6 minutes  
**Prerequisites:** `LESSON-D2-06`, `LESSON-D2-07`, `LAB-D2-03`; no prior PyTorch experience assumed  
**Environment:** CPU required; PyTorch 2.11-compatible APIs, NumPy, matplotlib, scikit-learn; 800 generated examples; no download

Workflow: **Map -> Predict -> Build baseline -> Inspect -> Diagnose evidence -> Reveal source -> Repair one invariant -> Compare -> Explain**. Restart and run in order. The first commitment cell intentionally stops execution.

In [ ]:
import platform
import random
import time

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
import torch
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device('cpu')
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.22})
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}")
print(f"Required device: {DEVICE}; optional GPU available: {torch.cuda.is_available()}")
print("The core path stays on CPU so every participant produces comparable evidence.")

## Current PyTorch Semantics Used Here

- `nn.BCEWithLogitsLoss` receives raw logits and same-shaped targets; it combines sigmoid and BCE with a numerically stable formulation.
- `loss.backward()` computes gradients and accumulates them into parameter `.grad` fields. It does not update parameters.
- `optimizer.step()` uses the current gradients to update parameters.
- `model.train()` and `model.eval()` control mode-sensitive modules such as dropout. `model.eval()` does **not** disable gradient recording.
- Evaluation therefore uses both `model.eval()` and `torch.inference_mode()` in this lab.
- The teaching path uses plain `optimizer.zero_grad()`, which in PyTorch 2.11 uses the default `set_to_none=True`: existing optimized gradients become `None`. Before `backward()`, inspection code must therefore handle `None`; afterward, parameters that received gradients hold tensors while parameters that received no gradient remain `None`. Optimizers treat `None` differently from a zero tensor: `None` skips that parameter's step, while a zero gradient can still trigger an update from optimizer state. Use `optimizer.zero_grad(set_to_none=False)` only when intentionally contrasting zero-filled gradient buffers.

## Recap: Map NumPy Responsibilities to Framework Calls

PyTorch automates derivative bookkeeping and parameter registration. It does not choose data, architecture, objective, modes, metric, or debugging evidence. Map each NumPy responsibility to the framework operation before writing code.

In [ ]:
framework_map = {
    "parameter_dictionary": "",
    "manual_forward": "",
    "clipped_BCE": "",
    "manual_backward": "",
    "manual_update": "",
    "evaluation_without_gradients": "",
}
assert all(value.strip() for value in framework_map.values()), (
    "Mapping checkpoint: connect every NumPy responsibility to a framework operation first."
)

## Local Dataset: Same Split as `LAB-D2-03`

Use the same 800-example moons generator, 25% stratified validation split, training-only standardization, and seeds. Converting to float32 tensors changes representation, not the split specification. Binary logits and targets both use shape `(B, 1)`.

In [ ]:
DATA_SEED = 23
MODEL_SEED = 24
BATCH_SEED = 25
X_all, y_all = make_moons(n_samples=800, noise=0.27, random_state=DATA_SEED)
X_train_raw, X_val_raw, y_train_flat, y_val_flat = train_test_split(
    X_all, y_all, test_size=0.25, random_state=DATA_SEED, stratify=y_all
)
train_mean = X_train_raw.mean(axis=0, keepdims=True)
train_std = X_train_raw.std(axis=0, keepdims=True)
X_train = (X_train_raw - train_mean) / train_std
X_val = (X_val_raw - train_mean) / train_std
y_train = y_train_flat.reshape(-1, 1).astype(np.float32)
y_val = y_val_flat.reshape(-1, 1).astype(np.float32)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32, device=DEVICE)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32, device=DEVICE)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32, device=DEVICE)
assert X_train_tensor.shape == (600, 2) and X_val_tensor.shape == (200, 2)
assert y_train_tensor.shape == (600, 1) and y_val_tensor.shape == (200, 1)
assert X_train_tensor.device.type == X_val_tensor.device.type == 'cpu'
print("Tensor shapes:", X_train_tensor.shape, y_train_tensor.shape, X_val_tensor.shape, y_val_tensor.shape)

In [ ]:
def make_train_loader(batch_size, seed):
    generator = torch.Generator().manual_seed(seed)
    dataset = TensorDataset(X_train_tensor, y_train_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=generator)

shape_loader = make_train_loader(64, BATCH_SEED)
first_X_batch, first_y_batch = next(iter(shape_loader))
assert first_X_batch.shape == (64, 2) and first_y_batch.shape == (64, 1)
print("First mini-batch shapes:", first_X_batch.shape, first_y_batch.shape)

## Predict the Baseline Contract

Predict the logits shape for 64 examples, the target shape required by `BCEWithLogitsLoss`, when sigmoid should enter the workflow, and the expected validation accuracy band. State separately what `backward`, `step`, `eval`, and `inference_mode` change.

In [ ]:
baseline_predictions = {
    "logits_shape": "",
    "target_shape": "",
    "sigmoid_location": "",
    "validation_accuracy_band": "",
    "backward_vs_step": "",
    "eval_vs_inference_mode": "",
}
assert all(value.strip() for value in baseline_predictions.values())

## Modify: Define a Logits Model

Complete `MoonClassifier`: `Linear(2, 8) -> Tanh -> Dropout(0.15) -> Linear(8, 1)`. Return raw logits. Do not apply sigmoid inside `forward`; probability conversion belongs to evaluation.

In [ ]:
class MoonClassifier(nn.Module):
    def __init__(self, hidden_dim=8):
        super().__init__()
        # TODO: register hidden, tanh, dropout, and output modules.
        raise NotImplementedError("TODO: define the module layers")

    def forward(self, X):
        # TODO: return raw logits with shape (B, 1).
        raise NotImplementedError("TODO: implement the logits forward pass")

In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_all_seeds(MODEL_SEED)
baseline_model = MoonClassifier().to(DEVICE)
sample_logits = baseline_model(first_X_batch)
loss_fn = nn.BCEWithLogitsLoss()
sample_loss = loss_fn(sample_logits, first_y_batch)
assert sample_logits.shape == first_y_batch.shape == (64, 1)
assert sample_logits.requires_grad and sample_loss.requires_grad
print("Sample logits/target/loss shapes:", sample_logits.shape, first_y_batch.shape, sample_loss.shape)

## Modify: Evaluation Has Two Independent Controls

Complete `evaluate_model`. Switch the module to evaluation mode, enter `torch.inference_mode()`, compute logits/loss, convert logits to probabilities with sigmoid, and return loss, accuracy, probabilities, and whether the returned logits require gradients.

In [ ]:
def evaluate_model(model, X, y, loss_fn):
    # TODO: use eval mode plus inference_mode, then return the requested evidence dictionary.
    raise NotImplementedError("TODO: implement gradient-disabled evaluation")

## Modify: Implement the Explicit Training Sequence

Complete `train_model`. At every mini-batch use this exact order: `optimizer.zero_grad()` -> forward -> loss -> `loss.backward()` -> record total gradient norm -> `optimizer.step()`. Call `model.train()` at the start of every epoch and the supplied evaluation helper afterward. With the default reset, expect optimized parameter `.grad` fields to be `None` before `backward()`; after `backward()`, inspect gradient tensors only for parameters that received a gradient, leaving others as `None`.

In [ ]:
def total_gradient_norm(model):
    squared = [torch.sum(parameter.grad.detach() ** 2) for parameter in model.parameters() if parameter.grad is not None]
    return float(torch.sqrt(torch.stack(squared).sum()).item()) if squared else 0.0

def train_model(model, *, learning_rate=0.6, epochs=300, batch_size=64, batch_seed=BATCH_SEED):
    # TODO: implement the explicit optimizer sequence and return a history dictionary.
    raise NotImplementedError("TODO: implement the PyTorch training loop")

## Predict Before Training

Predict whether parameters change immediately after `backward` or only after `step`. Predict the final train/validation relationship and a plausible gradient-norm pattern. Preserve your response while the baseline runs.

In [ ]:
training_predictions = {
    "parameter_change_moment": "",
    "train_validation_relationship": "",
    "gradient_norm_pattern": "",
}
assert all(value.strip() for value in training_predictions.values())

In [ ]:
set_all_seeds(MODEL_SEED)
baseline_model = MoonClassifier().to(DEVICE)
started = time.perf_counter()
baseline_history = train_model(baseline_model)
baseline_seconds = time.perf_counter() - started
baseline_train_evidence = evaluate_model(baseline_model, X_train_tensor, y_train_tensor, loss_fn)
baseline_val_evidence = evaluate_model(baseline_model, X_val_tensor, y_val_tensor, loss_fn)
print(f"Baseline runtime: {baseline_seconds:.3f}s")
print(f"Train loss/accuracy: {baseline_train_evidence['loss']:.4f} / {baseline_train_evidence['accuracy']:.3f}")
print(f"Validation loss/accuracy: {baseline_val_evidence['loss']:.4f} / {baseline_val_evidence['accuracy']:.3f}")
assert baseline_seconds < 120.0
assert 0.85 <= baseline_val_evidence['accuracy'] <= 0.95
assert baseline_val_evidence['logits_require_grad'] is False

## Inspect: Baseline Curves, Gradient Norms, and Boundary

These panels map framework evidence back to the NumPy dashboard. Loss and accuracy show outcome trajectories; gradient norm confirms backward produced a finite signal; the boundary shows the learned nonlinear region.

In [ ]:
def boundary_probabilities(model, mesh):
    model.eval()
    with torch.inference_mode():
        logits = model(torch.tensor(mesh, dtype=torch.float32, device=DEVICE))
        return torch.sigmoid(logits).cpu().numpy().ravel()

x1 = np.linspace(min(X_train[:, 0].min(), X_val[:, 0].min()) - 0.6, max(X_train[:, 0].max(), X_val[:, 0].max()) + 0.6, 180)
x2 = np.linspace(min(X_train[:, 1].min(), X_val[:, 1].min()) - 0.6, max(X_train[:, 1].max(), X_val[:, 1].max()) + 0.6, 180)
xx1, xx2 = np.meshgrid(x1, x2)
mesh = np.column_stack([xx1.ravel(), xx2.ravel()])
mesh_probability = boundary_probabilities(baseline_model, mesh).reshape(xx1.shape)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot(baseline_history['train_loss'], label='train', color='#277da1')
axes[0, 0].plot(baseline_history['val_loss'], label='validation', color='#f9844a')
axes[0, 0].set(title='BCEWithLogitsLoss', xlabel='epoch', ylabel='loss')
axes[0, 0].legend()
axes[0, 1].plot(baseline_history['train_accuracy'], label='train', color='#277da1')
axes[0, 1].plot(baseline_history['val_accuracy'], label='validation', color='#f9844a')
axes[0, 1].set(title='Accuracy', xlabel='epoch', ylabel='accuracy', ylim=(0.45, 1.0))
axes[0, 1].legend()
axes[1, 0].plot(baseline_history['gradient_norm'], color='#6a4c93')
axes[1, 0].set(title='Mean mini-batch gradient norm', xlabel='epoch', ylabel='L2 norm', yscale='log')
axes[1, 1].contourf(xx1, xx2, mesh_probability, levels=np.linspace(0, 1, 11), cmap='RdYlBu_r', alpha=0.45)
axes[1, 1].contour(xx1, xx2, mesh_probability, levels=[0.5], colors='black', linewidths=2)
for label, marker, color in [(0, 'o', '#277da1'), (1, '^', '#f9844a')]:
    mask = y_val_flat == label
    axes[1, 1].scatter(X_val[mask, 0], X_val[mask, 1], marker=marker, color=color, edgecolor='black', linewidth=0.3, s=30, label=f'class {label}')
axes[1, 1].set(title='Validation decision region', xlabel='standardized x1', ylabel='standardized x2')
axes[1, 1].legend()
plt.tight_layout()
plt.show()

## Model Detective: Evidence Before Faulty Source

Choose or receive one card. Do not inspect the later source fragments until you submit two observations, two plausible causes, the cheapest discriminating check, a predicted result, and a rejection condition. Curve symptoms are not unique fingerprints.

| Card | Evidence available now |
|---|---|
| A | Loss falls slightly then stalls near the linear baseline; gradients are finite; no shape error |
| B | Early loss falls; later values become jagged; gradient norms grow across mini-batches |
| C | Repeated validation passes disagree for identical examples; a mode-sensitive module is present |
| D | Shapes are valid, loss changes slowly, and probability quality looks inconsistent with the loss contract |

In [ ]:
assigned_card = ""  # TODO: A, B, C, or D.
evidence_only_diagnosis = {
    "two_observations": "",
    "two_plausible_causes": "",
    "cheapest_discriminating_check": "",
    "predicted_check_result": "",
    "rejection_condition": "",
}
assert assigned_card in {'A', 'B', 'C', 'D'}
assert all(value.strip() for value in evidence_only_diagnosis.values()), (
    "Evidence gate: submit the diagnosis before inspecting faulty source."
)

## Source Reveal: Four One-Invariant Configurations

Only now inspect the configurable model and training/evaluation harness. Each card violates exactly one invariant: compatible raw output/loss, hidden nonlinearity, gradient reset, or evaluation mode/context.

In [ ]:
class DiagnosticClassifier(nn.Module):
    def __init__(self, use_nonlinearity=True, output_probabilities=False):
        super().__init__()
        self.hidden = nn.Linear(2, 8)
        self.activation = nn.Tanh()
        self.dropout = nn.Dropout(0.15)
        self.output = nn.Linear(8, 1)
        self.use_nonlinearity = use_nonlinearity
        self.output_probabilities = output_probabilities

    def forward(self, X):
        hidden = self.hidden(X)
        activated = self.activation(hidden) if self.use_nonlinearity else hidden
        logits = self.output(self.dropout(activated))
        return torch.sigmoid(logits) if self.output_probabilities else logits

BASE_DIAGNOSTIC_CONFIG = {
    "use_nonlinearity": True,
    "output_probabilities": False,
    "reset_gradients": True,
    "proper_evaluation": True,
}
CARD_TO_DEFECT = {
    "A": 'missing_nonlinearity',
    "B": 'omitted_gradient_reset',
    "C": 'mode_misuse',
    "D": 'wrong_output_loss',
}
assigned_defect = CARD_TO_DEFECT[assigned_card]

In [ ]:
def defect_configuration(defect):
    config = BASE_DIAGNOSTIC_CONFIG.copy()
    if defect == 'missing_nonlinearity':
        config['use_nonlinearity'] = False
    elif defect == 'omitted_gradient_reset':
        config['reset_gradients'] = False
    elif defect == 'mode_misuse':
        config['proper_evaluation'] = False
    elif defect == 'wrong_output_loss':
        config['output_probabilities'] = True
    else:
        raise ValueError(f"Unknown defect: {defect}")
    return config

def diagnostic_evaluate(model, config):
    if config['proper_evaluation']:
        model.eval()
        with torch.inference_mode():
            outputs = [model(X_val_tensor) for _ in range(3)]
    else:
        model.train()
        outputs = [model(X_val_tensor) for _ in range(3)]
    first_output = outputs[0]
    probabilities = torch.sigmoid(first_output)
    stacked = torch.stack([torch.sigmoid(output.detach()) for output in outputs])
    return {
        "loss": float(nn.BCEWithLogitsLoss()(first_output, y_val_tensor).detach().item()),
        "accuracy": float(((probabilities >= 0.5) == y_val_tensor.bool()).float().mean().detach().item()),
        "repeat_std": float(stacked.std(dim=0).mean().item()),
        "outputs_require_grad": bool(first_output.requires_grad),
    }

def run_diagnostic(config, epochs=180):
    set_all_seeds(MODEL_SEED)
    model = DiagnosticClassifier(config['use_nonlinearity'], config['output_probabilities']).to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.6)
    criterion = nn.BCEWithLogitsLoss()
    loader = make_train_loader(64, BATCH_SEED)
    history = {"train_loss": [], "gradient_norm": [], "val_loss": [], "val_accuracy": []}
    for _ in range(epochs):
        model.train()
        losses, norms = [], []
        for X_batch, y_batch in loader:
            if config['reset_gradients']:
                optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            norms.append(total_gradient_norm(model))
            optimizer.step()
            losses.append(float(loss.detach().item()))
        evidence = diagnostic_evaluate(model, config)
        history['train_loss'].append(float(np.mean(losses)))
        history['gradient_norm'].append(float(np.mean(norms)))
        history['val_loss'].append(evidence['loss'])
        history['val_accuracy'].append(evidence['accuracy'])
    return model, history, evidence

## Challenge: Repair Exactly One Invariant

Complete `repair_configuration`. Start from the assigned faulty configuration and change only the field that restores its violated invariant. Predict the evidence change before either run. The check rejects broad rewrites that change more than one field.

In [ ]:
broken_config = defect_configuration(assigned_defect)
repair_prediction = {"curve": "", "gradient_norm": "", "accuracy_or_repeatability": "", "mechanism": ""}
assert all(value.strip() for value in repair_prediction.values())

def repair_configuration(defect, broken_config):
    # TODO: copy the config and restore only the invariant violated by `defect`.
    raise NotImplementedError("TODO: implement one targeted repair")

repaired_config = repair_configuration(assigned_defect, broken_config)
changed_fields = [name for name in broken_config if broken_config[name] != repaired_config[name]]
expected_field = {
    'missing_nonlinearity': 'use_nonlinearity',
    'omitted_gradient_reset': 'reset_gradients',
    'mode_misuse': 'proper_evaluation',
    'wrong_output_loss': 'output_probabilities',
}[assigned_defect]
assert changed_fields == [expected_field]
assert repaired_config == BASE_DIAGNOSTIC_CONFIG

In [ ]:
broken_model, broken_history, broken_evidence = run_diagnostic(broken_config)
repaired_model, repaired_history, repaired_evidence = run_diagnostic(repaired_config)
print("Assigned defect:", assigned_defect)
print("Broken evidence:", broken_evidence)
print("Repaired evidence:", repaired_evidence)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(broken_history['val_loss'], label='broken', color='#d1495b')
axes[0].plot(repaired_history['val_loss'], label='repaired', color='#277da1')
axes[0].set(title='Validation loss', xlabel='epoch', ylabel='loss')
axes[0].legend()
axes[1].plot(broken_history['gradient_norm'], label='broken', color='#d1495b')
axes[1].plot(repaired_history['gradient_norm'], label='repaired', color='#277da1')
axes[1].set(title='Gradient norm', xlabel='epoch', ylabel='L2 norm', yscale='log')
axes[1].legend()
axes[2].plot(broken_history['val_accuracy'], label='broken', color='#d1495b')
axes[2].plot(repaired_history['val_accuracy'], label='repaired', color='#277da1')
axes[2].set(title='Validation accuracy', xlabel='epoch', ylabel='accuracy', ylim=(0.45, 1.0))
axes[2].legend()
plt.tight_layout()
plt.show()
assert 0.85 <= repaired_evidence['accuracy'] <= 0.95
assert repaired_evidence['repeat_std'] < 1e-8
assert repaired_evidence['outputs_require_grad'] is False

## Explain the Repair

Cite the symptom, competing hypothesis, discriminating check, violated invariant, one-field repair, before/after evidence, and mechanism. State one remaining uncertainty: the repaired classroom run does not make every larger training symptom uniquely diagnosable.

In [ ]:
repair_diagnosis = {
    "symptom": "",
    "competing_hypothesis": "",
    "discriminating_check": "",
    "violated_invariant": "",
    "targeted_repair": "",
    "before_after_evidence": "",
    "why_it_worked": "",
    "remaining_uncertainty": "",
}
assert all(value.strip() for value in repair_diagnosis.values())

## Inspect the Mode/Gradient Distinction Directly

Even if your assigned card was different, verify this semantic distinction: `eval()` changes dropout behavior, while gradient recording remains enabled until a no-grad or inference context is entered. Predict both `requires_grad` values before running.

In [ ]:
mode_prediction = {"eval_only_requires_grad": "", "inference_context_requires_grad": ""}
assert all(value.strip() for value in mode_prediction.values())
baseline_model.eval()
eval_only_logits = baseline_model(X_val_tensor[:8])
with torch.inference_mode():
    inference_logits = baseline_model(X_val_tensor[:8])
print("eval() only -> requires_grad:", eval_only_logits.requires_grad)
print("eval() + inference_mode() -> requires_grad:", inference_logits.requires_grad)
assert eval_only_logits.requires_grad is True
assert inference_logits.requires_grad is False

## Optional Extension: Shape Mismatch Diagnostic

Predict the exception category when logits have shape `(B, 1)` but targets are flattened to `(B,)`. Run the mismatch inside a caught block, then recover by restoring the explicit target dimension. This extension is not required by the core checkpoint.

In [ ]:
optional_shape_prediction = ""
if not optional_shape_prediction.strip():
    print("Optional shape diagnostic skipped. Core checkpoint is unaffected.")
else:
    baseline_model.eval()
    with torch.inference_mode():
        optional_logits = baseline_model(X_val_tensor[:16])
    shape_failure_observed = False
    try:
        loss_fn(optional_logits, y_val_tensor[:16].ravel())
    except ValueError as error:
        shape_failure_observed = True
        print("Expected shape mismatch:", error)
    recovered_loss = loss_fn(optional_logits, y_val_tensor[:16])
    assert shape_failure_observed and torch.isfinite(recovered_loss)
    optional_shape_interpretation = ""  # TODO (optional): explain why same element count is insufficient.
    assert optional_shape_interpretation.strip()

## Reflect and Checkpoint

Map every framework call back to a NumPy responsibility. Explain what autograd automated, what judgment remained yours, and which evidence would catch the assigned defect in a larger run.

In [ ]:
reflection = {
    "framework_to_numpy_mapping": "",
    "autograd_automated": "",
    "judgment_remained": "",
    "larger_run_detection": "",
}
assert all(value.strip() for value in reflection.values())
assert 0.85 <= baseline_val_evidence['accuracy'] <= 0.95
assert 0.85 <= repaired_evidence['accuracy'] <= 0.95
assert changed_fields == [expected_field]
assert repaired_evidence['repeat_std'] < 1e-8
assert repaired_evidence['outputs_require_grad'] is False
assert eval_only_logits.requires_grad and not inference_logits.requires_grad
assert baseline_seconds < 120.0
print("LAB-D2-04 checkpoint passed: logits/loss contract, optimizer order, mode semantics, gradient evidence, and targeted repair.")

## Takeaways

- `BCEWithLogitsLoss` pairs raw logits with same-shaped binary targets.
- `backward()` computes and accumulates gradients; `step()` updates parameters.
- Gradient reset belongs before each new mini-batch backward pass unless accumulation is intentional.
- `eval()` controls mode-sensitive modules; an inference context controls gradient recording.
- PyTorch automates mechanics, but evidence-first diagnosis and objective choice remain engineering decisions.

## Troubleshooting

| Symptom | Likely cause | Recovery |
|---|---|---|
| `ModuleNotFoundError: torch` | Selected kernel lacks PyTorch | Select the course kernel or install the course-compatible PyTorch build, then restart |
| Loss rejects target size | Logits `(B,1)` and targets `(B,)` differ | Preserve the explicit binary output dimension |
| Loss behaves oddly with values in `[0,1]` | Sigmoid was applied before `BCEWithLogitsLoss` | Return raw logits and apply sigmoid only for metrics |
| Gradient norms grow across batches | Gradients were not reset | Call plain `optimizer.zero_grad()` before the next backward pass |
| Validation outputs change between repeats | Model remained in training mode with dropout active | Call `model.eval()` before evaluation |
| Evaluation logits still require gradients | `eval()` was mistaken for gradient disabling | Add `torch.inference_mode()` or `torch.no_grad()` |
| Results differ from the sanity band | Seeds, split, mode, or device changed | Restart, restore CPU and supplied constants, and run in order |

## Continue

Return to the [LAB-D2-04 debrief](../student-guide/day-2-student-guide.md#lab-d2-04---pytorch-autograd-break-it-and-fix-it). Review [LESSON-D2-06](../student-guide/day-2-student-guide.md#lesson-d2-06---what-pytorch-automates), [LESSON-D2-07](../student-guide/day-2-student-guide.md#lesson-d2-07---evidence-first-training-diagnosis), [ACT-D2-04](../challenges/day-2-challenges.md#act-d2-04---broken-curve-detective), and [LAB-D2-03](LAB-D2-03-numpy-training.ipynb) as needed.